In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm
from warnings import filterwarnings
filterwarnings('ignore')

source_lbl_dir = r'..\data\RDD_2024\RDD2024_ChinaJapanIndia\labels'
filtered_lbl_dir = '../data/RDD_2024/Filtered_Dataset_RDD2024_China_Japan_India_6_classes/labels'
balanced_lbl_dir = '../data/RDD_2024/Balanced_Dataset_Final_china_japan_india/labels'

# Nama kelas asli (Sesuai dokumentasi RDD)
original_names = {
    '0': 'Longitudinal Crack', '1': 'Transverse Crack', '2': 'Alligator Crack',
    '3': 'Repaired Crack', '4': 'Pothole', '5': 'Crossing Blur',
    '6': 'Lane Line Blur', '7': 'Manhole', '8': 'Patchy Road', '9': 'Rutting'
}

# Nama kelas baru setelah filtering/mapping (5 Kelas)
# new_names = {
#     '0': 'Linear_Crack',
#     '1': 'Alligator_Crack',
#     '2': 'Pothole',
#     '3': 'Manhole',
#     '4': 'Patchy_Road'
# }

# Nama kelas baru setelah filtering/mapping (6 Kelas)
new_names = {
    '0': 'Longitudinal_Crack',
    '1': 'Transverse_Crack',
    '2': 'Alligator_Crack',
    '3': 'Pothole',
    '4': 'manhole',
    '5': 'patchy_road'
}


In [ ]:
def get_distribution(label_path):
    all_classes = []
    files = [f for f in os.listdir(label_path) if f.endswith('.txt')]
    for file in tqdm(files, desc="Processing labels"):
        with open(os.path.join(label_path, file), 'r') as f:
            for line in f:
                cls = line.split()[0]
                all_classes.append(cls)
    return Counter(all_classes)

dist_original = get_distribution(source_lbl_dir)
dist_filtered = get_distribution(filtered_lbl_dir)
# dist_balanced = get_distribution(balanced_lbl_dir)

In [ ]:
# Convert ke DataFrame untuk memudahkan plotting
df_orig = pd.DataFrame.from_dict(dist_original, orient='index', columns=['Count']).reset_index()
df_orig['Class_Name'] = df_orig['index'].map(original_names)
df_orig = df_orig.sort_values(by='index')

df_filt = pd.DataFrame.from_dict(dist_filtered, orient='index', columns=['Count']).reset_index()
df_filt['Class_Name'] = df_filt['index'].map(new_names)
df_filt = df_filt.sort_values(by='index')

# df_balanced = pd.DataFrame.from_dict(dist_balanced, orient='index', columns=['Count']).reset_index()
# df_balanced['Class_Name'] = df_balanced['index'].map(new_names)
# df_balanced = df_balanced.sort_values(by='index')

In [ ]:
# Plotting
plt.figure(figsize=(22, 6))

# Plot 1: Original Distribution
plt.subplot(1, 3, 1)
original_order = [original_names[key] for key in sorted(original_names.keys(), key=int)]
sns.barplot(data=df_orig, x='Class_Name', y='Count', order=original_order, hue='Class_Name', palette='viridis', legend=False)
plt.title('Original Class Distribution (10 Classes)')
plt.xlabel('Class Name')
plt.xticks(rotation=45, ha='right')

# Plot 2: Filtered Distribution
plt.subplot(1, 3, 2)
sns.barplot(data=df_filt, x='Class_Name', y='Count', hue='Class_Name', palette='magma', legend=False)
plt.title('Class Distribution After Filtering & Mapping')
plt.xticks(rotation=15)

# Plot 3: Balanced Distribution
plt.subplot(1, 3, 3)
sns.barplot(data=df_balanced, x='Class_Name', y='Count', hue='Class_Name', palette='crest', legend=False)
plt.title('Class Distribution After Balancing')
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

print(f"Total Objek di Dataset Asli: {sum(dist_original.values())}")
print(f"Total Objek di Dataset Filtered: {sum(dist_filtered.values())}")
print(f"Total Objek di Dataset Balanced: {sum(dist_balanced.values())}")

In [ ]:
# Plot khusus Original Dataset (Before Filtering)
plt.figure(figsize=(14, 6))

# Original Distribution Detail
original_order = [original_names[key] for key in sorted(original_names.keys(), key=int)]
sns.barplot(data=df_orig, x='Class_Name', y='Count', hue='Class_Name',
            order=original_order, palette='Set2', legend=False)

plt.title('Class Distribution - Original Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Class Name', fontsize=12)
plt.ylabel('Number of Objects', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add values on top of each bar
ax = plt.gca()
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height,
            f'{int(height)}',
            ha="center", va="bottom", fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Print statistics detail
print("=" * 50)
print("ORIGINAL DATASET STATISTICS")
print("=" * 50)
for idx, row in df_orig.iterrows():
    print(f"{row['Class_Name']:25} : {row['Count']:6} objects")

## PLOT FILTERED DATASET

In [ ]:
# Distribusi object per negara (berdasarkan prefix filename)
from collections import defaultdict

def get_country_from_filename(filename):
    name = filename.lower()
    if name.startswith('china_'):
        return 'China'
    if name.startswith('india_'):
        return 'India'
    if name.startswith('japan_'):
        return 'Japan'
    return None

rows = []
for file in os.listdir(filtered_lbl_dir):
    if not file.endswith('.txt') or file == 'classes.txt':
        continue
    country = get_country_from_filename(file)
    if country is None:
        continue
    with open(os.path.join(filtered_lbl_dir, file), 'r') as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            rows.append({
                'Country': country,
                'Class': parts[0]
            })

df_country = pd.DataFrame(rows)

In [ ]:
# Pie chart jumlah gambar per negara
from pathlib import Path

filtered_img_dir = Path('../data/RDD_2024/Filtered_Dataset_RDD2024_China_Japan_India_6_classes/images')
valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG'}

if not filtered_img_dir.exists():
    print('Folder images tidak ditemukan:', filtered_img_dir)
else:
    images = [p for p in filtered_img_dir.iterdir() if p.suffix in valid_ext]
    if not images:
        print('Tidak ada gambar untuk divisualisasikan.')
    else:
        counts = {'China': 0, 'India': 0, 'Japan': 0}
        for img in images:
            name = img.name.lower()
            if name.startswith('china_'):
                counts['China'] += 1
            elif name.startswith('india_'):
                counts['India'] += 1
            elif name.startswith('japan_'):
                counts['Japan'] += 1

        country_order = ['China', 'India', 'Japan']
        country_counts = pd.Series([counts[c] for c in country_order], index=country_order)
        colors = ['#ffcc66', '#66c2a5', '#8da0cb']
        total_images = int(country_counts.sum())
        labels = [f"{c} ({int(country_counts[c])})" for c in country_counts.index]

        plt.figure(figsize=(7.5, 7.5))
        wedges, texts, autotexts = plt.pie(
            country_counts.values,
            labels=labels,
            autopct='%1.1f%%',
            startangle=140,
            colors=colors,
            wedgeprops={'edgecolor': 'white'}
        )
        plt.title(
            f"Image Distribution by Country (Filtered Dataset)\nTotal: {total_images:,}",
            fontsize=14,
            fontweight='bold'
        )
        for t in texts + autotexts:
            t.set_fontsize(11)
        plt.tight_layout()
        plt.show()

        print('Jumlah gambar per negara:')
        display(country_counts.to_frame(name='Count'))

In [ ]:
# Plot khusus Filtered Dataset
plt.figure(figsize=(12, 6))

# Filtered Distribution Detail
filtered_order = [new_names[key] for key in sorted(new_names.keys(), key=int)]
sns.barplot(data=df_filt, x='Class_Name', y='Count', hue='Class_Name',
            order=filtered_order, palette='husl', legend=False)

plt.title('Class Distribution - Filtered Dataset', fontsize=14, fontweight='bold')
plt.xlabel('Class Name', fontsize=12)
plt.ylabel('Number of Objects', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Tambahkan nilai di atas setiap bar
ax = plt.gca()
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height,
            f'{int(height)}',
            ha="center", va="bottom", fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Print statistik detail
print("=" * 50)
print("FILTERED DATASET STATISTICS")
print("=" * 50)
for idx, row in df_filt.iterrows():
    print(f"{row['Class_Name']:20} : {row['Count']:6} objects")

In [ ]:
if df_country.empty:
    print('Tidak ada data label untuk divisualisasikan.')
else:
    df_country['Class_Name'] = df_country['Class'].map(new_names)
    class_order = [new_names[key] for key in sorted(new_names.keys(), key=int)]
    country_order = ['China', 'India', 'Japan']

    plt.figure(figsize=(14, 6))
    ax = sns.countplot(
        data=df_country,
        x='Class_Name',
        hue='Country',
        order=class_order,
        hue_order=country_order,
        palette='Set2'
    )
    plt.title('Object Distribution per Country (Filtered Dataset)', fontsize=14, fontweight='bold')
    plt.xlabel('Class Name', fontsize=12)
    plt.ylabel('Number of Objects', fontsize=12)
    plt.xticks(rotation=45, ha='right')

    # Tampilkan nilai di atas bar
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.text(
                p.get_x() + p.get_width() / 2,
                height,
                f'{int(height)}',
                ha='center',
                va='bottom',
                fontsize=9
            )

    plt.tight_layout()
    plt.show()

    # Tabel ringkas per negara dan kelas
    summary = (
        df_country
        .groupby(['Country', 'Class_Name'])
        .size()
        .unstack(fill_value=0)
        .reindex(index=country_order, columns=class_order, fill_value=0)
    )
    print('Ringkasan jumlah object per negara:')
    display(summary)

In [ ]:
print("Original Class Distribution:")
print(dist_original)
print("\nFiltered Class Distribution:")
print(dist_filtered)
# print("\nBalanced Class Distribution:")
# print(dist_balanced)

In [ ]:
# Contoh visualisasi bbox per label (Filtered Dataset)
from pathlib import Path
import matplotlib.patches as patches
import random
import math

random.seed(42)

filtered_root = Path('../data/RDD_2024/Filtered_Dataset_RDD2024_China_Japan_India_6_classes')
images_dir = filtered_root / 'images'
labels_dir = filtered_root / 'labels'

valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG'}
class_map = new_names
class_ids = sorted(class_map.keys(), key=int)
rows = 2
cols = math.ceil(len(class_ids) / rows)

label_code_map = {
    '0': 'D00',
    '1': 'D10',
    '2': 'D20',
    '3': 'D40',
    '4': 'D70',
    '5': 'D80'
}

def find_image_for_label(label_file: Path, img_dir: Path):
    stem = label_file.stem
    for ext in valid_ext:
        candidate = img_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    fallback = list(img_dir.glob(f"{stem}.*"))
    return fallback[0] if fallback else None

def yolo_to_xyxy(xc, yc, bw, bh, img_w, img_h):
    x1 = (xc - bw / 2) * img_w
    y1 = (yc - bh / 2) * img_h
    x2 = (xc + bw / 2) * img_w
    y2 = (yc + bh / 2) * img_h
    return x1, y1, x2, y2

# Kumpulkan kandidat per kelas (satu contoh terbaik)
candidates = []
for label_file in sorted(labels_dir.glob('*.txt')):
    if label_file.name == 'classes.txt':
        continue
    if label_file.stat().st_size == 0:
        continue
    image_path = find_image_for_label(label_file, images_dir)
    if image_path is None:
        continue
    boxes = []
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id, xc, yc, bw, bh = parts
            if cls_id not in class_map:
                continue
            boxes.append({
                'cls_id': cls_id,
                'bbox': tuple(map(float, [xc, yc, bw, bh]))
            })
    if not boxes:
        continue
    candidates.append({
        'label_file': label_file,
        'image_path': image_path,
        'boxes': boxes
    })

samples_per_class = {cls_id: None for cls_id in class_ids}
for cls_id in class_ids:
    eligible = []
    for item in candidates:
        target_boxes = [b for b in item['boxes'] if b['cls_id'] == cls_id]
        if not target_boxes:
            continue
        eligible.append({
            **item,
            'target_boxes': target_boxes,
            'target_box_count': len(target_boxes),
            'total_box_count': len(item['boxes'])
        })
    eligible.sort(key=lambda x: (x['target_box_count'], x['total_box_count']), reverse=True)
    if eligible:
        samples_per_class[cls_id] = eligible[0]

fig, axes = plt.subplots(rows, cols, figsize=(5.5 * cols, 5 * rows))
if rows == 1 and cols == 1:
    axes = [[axes]]
elif rows == 1:
    axes = [axes]
elif cols == 1:
    axes = [[ax] for ax in axes]

color_map = {
    '0': '#66c2a5',
    '1': '#fc8d62',
    '2': '#8da0cb',
    '3': '#e78ac3',
    '4': '#a6d854',
    '5': '#ffd92f'
}

for idx, cls_id in enumerate(class_ids):
    r = idx // cols
    c = idx % cols
    ax = axes[r][c]
    sample = samples_per_class[cls_id]
    if sample is None:
        ax.text(0.5, 0.5, 'Sample not found', ha='center', va='center', fontsize=11)
        ax.set_title(f"Class {cls_id} ({class_map[cls_id]})")
        ax.axis('off')
        continue
    img = plt.imread(sample['image_path'])
    h, w = img.shape[:2]
    ax.imshow(img)
    for box in sample['target_boxes']:
        xc, yc, bw, bh = box['bbox']
        x1, y1, x2, y2 = yolo_to_xyxy(xc, yc, bw, bh, w, h)
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1,
            edgecolor=color_map.get(cls_id, 'white'),
            facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1,
            max(0, y1 - 5),
            label_code_map.get(cls_id, cls_id),
            color='black',
            fontsize=8,
            bbox=dict(facecolor=color_map.get(cls_id, 'white'), alpha=0.85, edgecolor='none', pad=1)
        )
    ax.set_title(f"Class {cls_id} ({class_map[cls_id]})")
    ax.axis('off')

# Matikan axis yang kosong
total_slots = rows * cols
for idx in range(len(class_ids), total_slots):
    r = idx // cols
    c = idx % cols
    axes[r][c].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Analisis Ukuran Objek (Small Object Challenge) - Filtered Dataset
import numpy as np
from pathlib import Path
from tqdm import tqdm

filtered_root = Path('../data/RDD_2024/Filtered_Dataset_RDD2024_China_Japan_India_6_classes')
images_dir = filtered_root / 'images'
labels_dir = filtered_root / 'labels'

valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG'}

def find_image_for_label(label_file: Path, img_dir: Path):
    stem = label_file.stem
    for ext in valid_ext:
        candidate = img_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    fallback = list(img_dir.glob(f"{stem}.*"))
    return fallback[0] if fallback else None

bbox_area_pct = []
image_sizes = []

label_files = sorted(labels_dir.glob('*.txt'))
for label_file in tqdm(label_files, desc="Parsing bboxes"):
    if label_file.name == 'classes.txt':
        continue
    if label_file.stat().st_size == 0:
        continue
    img_path = find_image_for_label(label_file, images_dir)
    if img_path is None:
        continue
    img = plt.imread(img_path)
    h, w = img.shape[:2]
    img_area = float(w * h)
    if img_area == 0:
        continue
    image_sizes.append(img_area)
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, xc, yc, bw, bh = parts
            bw = float(bw)
            bh = float(bh)
            bbox_area = (bw * w) * (bh * h)
            bbox_area_pct.append((bbox_area / img_area) * 100.0)

bbox_area_pct = np.array(bbox_area_pct)
if bbox_area_pct.size == 0:
    print('Tidak ada bounding box untuk dianalisis.')
else:
    # Histogram persentase luas bbox
    plt.figure(figsize=(10, 5))
    plt.hist(bbox_area_pct, bins=50, color='#4c78a8', edgecolor='white')
    plt.title('Distribusi Persentase Luas Bounding Box (Filtered Dataset)')
    plt.xlabel('BBox Area (%)')
    plt.ylabel('Jumlah BBox')
    plt.tight_layout()
    plt.show()

    # Scatter: index vs persentase luas bbox
    plt.figure(figsize=(10, 5))
    plt.scatter(np.arange(len(bbox_area_pct)), bbox_area_pct, s=8, alpha=0.5, color='#f58518')
    plt.title('Scatter Persentase Luas Bounding Box (Filtered Dataset)')
    plt.xlabel('Index BBox')
    plt.ylabel('BBox Area (%)')
    plt.tight_layout()
    plt.show()

    print(f'Total bbox: {len(bbox_area_pct):,}')
    print(f'Median bbox area (%): {np.median(bbox_area_pct):.4f}')
    print(f'P90 bbox area (%): {np.percentile(bbox_area_pct, 90):.4f}')

In [ ]:
import random
import matplotlib.patches as patches
from pathlib import Path

# Setup paths
balanced_root = Path('../data/RDD_2024/Filtered_Dataset_RDD2024_China_Japan_India_6_classes')
images_dir = balanced_root / 'images'
labels_dir = balanced_root / 'labels'

# Valid image extensions
valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG'}

label_code_map = {
    '0': 'D00',
    '1': 'D10',
    '2': 'D20',
    '3': 'D40',
    '4': 'D70',
    '5': 'D80'
}

def find_image(label_file, img_dir):
    """Cari image yang sesuai dengan label file"""
    stem = label_file.stem
    for ext in valid_ext:
        candidate = img_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    fallback = list(img_dir.glob(f"{stem}.*"))
    return fallback[0] if fallback else None

def yolo_to_xyxy(xc, yc, bw, bh, img_w, img_h):
    """Convert YOLO format ke pixel coordinates"""
    x1 = (xc - bw / 2) * img_w
    y1 = (yc - bh / 2) * img_h
    x2 = (xc + bw / 2) * img_w
    y2 = (yc + bh / 2) * img_h
    return x1, y1, x2, y2

# Warna untuk setiap kelas
color_map = {
    '0': '#00b3ff',
    '1': '#fc8d62',
    '2': '#8da0cb',
    '3': '#e78ac3',
    '4': '#a6d854',
    '5': '#ffd92f'
}

# Cari gambar dengan banyak labels, grouped by country
print("🔍 Mencari gambar terbaik dari setiap negara...")
samples_by_country = {'China': [], 'India': [], 'Japan': []}

for label_file in sorted(labels_dir.glob('*.txt')):
    if label_file.name == 'classes.txt':
        continue

    if label_file.stat().st_size == 0:
        continue

    img_path = find_image(label_file, images_dir)
    if not img_path or not img_path.exists():
        continue

    # Tentukan negara
    filename = img_path.name.lower()
    country = None
    if filename.startswith('china_'):
        country = 'China'
    elif filename.startswith('india_'):
        country = 'India'
    elif filename.startswith('japan_'):
        country = 'Japan'
    else:
        continue

    # Hitung jumlah objects dan unique classes
    obj_count = 0
    unique_classes = set()

    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                obj_count += 1
                unique_classes.add(parts[0])

    if obj_count > 0:
        samples_by_country[country].append({
            'label_file': label_file,
            'img_path': img_path,
            'obj_count': obj_count,
            'class_count': len(unique_classes),
            'classes': unique_classes
        })

# Sort setiap negara by class_count dan obj_count
for country in samples_by_country:
    samples_by_country[country].sort(key=lambda x: (x['class_count'], x['obj_count']), reverse=True)

# Ambil yang terbaik dari masing-masing negara
selected = []
for country in ['China', 'India', 'Japan']:
    if samples_by_country[country]:
        selected.append(samples_by_country[country][0])
        sample = samples_by_country[country][0]
        classes_str = ', '.join([f"Class {c}" for c in sorted(sample['classes'])])
        print(f"   {country:10} | Objects: {sample['obj_count']:2} | Classes: {sample['class_count']} | {classes_str}")

# Plot 3 gambar
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

countries_order = ['China', 'India', 'Japan']

for idx, country in enumerate(countries_order):
    ax = axes[idx]

    # Cari sample untuk negara ini
    sample = None
    for s in selected:
        filename = s['img_path'].name.lower()
        if filename.startswith(country.lower() + '_'):
            sample = s
            break

    if not sample:
        ax.text(0.5, 0.5, f'No data for {country}', ha='center', va='center', fontsize=12)
        ax.set_title(f"{country} - No samples", fontsize=12)
        ax.axis('off')
        continue

    # Load dan display image
    img = plt.imread(sample['img_path'])
    ax.imshow(img)
    h, w = img.shape[:2]

    # Read dan plot bounding boxes
    with open(sample['label_file'], 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls_id, xc, yc, bw, bh = parts
            xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)

            # Convert ke pixel coordinates
            x1, y1, x2, y2 = yolo_to_xyxy(xc, yc, bw, bh, w, h)

            # Draw rectangle
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2,
                edgecolor=color_map.get(cls_id, 'white'),
                facecolor='none'
            )
            ax.add_patch(rect)

            # Add label text (pakai kode Dxx)
            ax.text(
                x1, max(0, y1 - 8),
                label_code_map.get(cls_id, cls_id),
                color='black',
                fontsize=10,
                fontweight='bold',
                bbox=dict(
                    facecolor=color_map.get(cls_id, 'white'),
                    alpha=0.85,
                    edgecolor='none',
                    pad=2
                )
            )

    # Title
    classes_in_img = ', '.join(sorted(sample['classes']))
    ax.set_title(
        f"{country} | {sample['obj_count']} Objects | {sample['class_count']} Classes\n"
        f"Classes: {classes_in_img}",
        fontsize=12,
        fontweight='bold',
        pad=10
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n✅ Selesai! Satu gambar terbaik dari setiap negara")
print("\n📝 Legenda Warna Bounding Box:")
for cls_id in sorted(new_names.keys()):
    print(f"   {cls_id:2} → {color_map[cls_id]} box")